# Session 10: Using Ragas to Evaluate a RAG Application built with LangChain and LangGraph

In the following notebook, we'll be looking at how [Ragas](https://github.com/explodinggradients/ragas) can be helpful in a number of ways when looking to evaluate your RAG applications!

While this example is rooted in LangChain/LangGraph - Ragas is framework agnostic (you don't even need to be using a framework!).

## 🤝 Breakout Room #1
  - Task 1: Installing Required Libraries
  - Task 2: Set Environment Variables
  - Task 3: Synthetic Dataset Generation for Evaluation using Ragas
  - Task 4: Construct our RAG application
  - Task 5: Evaluating our Application with Ragas
  - Task 6: Making Adjustments and Re-Evaluating
  - ***Activity #1: Implement a Different Reranking Strategy***


## Task 1: Installing Required Libraries

If you have not already done so, install the required libraries using the uv package manager:
``` bash

uv sync

```


## Task 2: Set Environment Variables:

We'll also need to provide our API keys.
> NOTE: In addition to OpenAI's models, this notebook will be using Cohere's Reranker - please be sure to [sign-up for an API key!](https://docs.cohere.com/reference/about)

You have two options for supplying your API keys in this session:
- Use environment variables (see Prerequisite #2 in the README.md)
- Provide them via a prompt when the notebook runs

The following code will load all of the environment variables in your `.env`. Then, it checks for the two API keys we need. If they are not there, it will prompt you to provide them.

First, OpenAI's for our LLM/embedding model combination!

Second, Cohere's for our reranking


In [1]:
import os
from getpass import getpass
from dotenv import load_dotenv

load_dotenv()

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Please enter your OpenAI API key!")

if not os.environ.get("COHERE_API_KEY"):
    os.environ["COHERE_API_KEY"] = getpass("Please enter your Cohere API key!")

## Task 3: Synthetic Dataset Generation for Evaluation using Ragas

We wil be using Ragas to build out a set of synthetic test questions, references, and reference contexts. This is useful because it will allow us to find out how our system is performing.

> NOTE: Ragas is best suited for finding *directional* changes in your LLM-based systems. The absolute scores aren't comparable in a vacuum.

### Data Preparation

We'll prepare our data using the Health & Wellness Guide - a comprehensive resource covering exercise, nutrition, sleep, and stress management.

Next, let's load our data into a familiar LangChain format using the `TextLoader`.

In [2]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader("data/HealthWellnessGuide.txt")
docs = loader.load()

### Knowledge Graph Based Synthetic Generation

Ragas uses a knowledge graph based approach to create data. This is extremely useful as it allows us to create complex queries rather simply. The additional testset complexity allows us to evaluate larger problems more effectively, as systems tend to be very strong on simple evaluation tasks.

Let's start by defining our `generator_llm` (which will generate our questions, summaries, and more), and our `generator_embeddings` which will be useful in building our graph.

### Abstracted SDG

The above method is the full process - but we can shortcut that using the provided abstractions!

This will generate our knowledge graph under the hood, and will - from there - generate our personas and scenarios to construct our queries.



In [3]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

In [4]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
dataset = generator.generate_with_langchain_docs(docs, testset_size=10)

Applying HeadlinesExtractor:   0%|          | 0/1 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/1 [00:00<?, ?it/s]

Applying SummaryExtractor:   0%|          | 0/1 [00:00<?, ?it/s]

Applying CustomNodeFilter:   0%|          | 0/4 [00:00<?, ?it/s]

Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/9 [00:00<?, ?it/s]

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/1 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/2 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/11 [00:00<?, ?it/s]

In [5]:
dataset.to_pandas()

,user_input,reference_contexts,reference,synthesizer_name
0,How can the Bird Dog exercise be incorporated ...,[The Personal Wellness Guide A Comprehensive R...,The Bird Dog exercise can be included in a wel...,single_hop_specifc_query_synthesizer
1,Wut is the Personal Wellness Guide and how can...,[The Personal Wellness Guide A Comprehensive R...,The Personal Wellness Guide is a comprehensive...,single_hop_specifc_query_synthesizer
2,Chamomile tea help sleep? How use for better s...,[PART 3: SLEEP AND RECOVERY Chapter 7: The Sci...,Herbal teas such as chamomile are listed as na...,single_hop_specifc_query_synthesizer
3,Cud yu pleese explane wut Chapter 8 sez abot i...,[PART 3: SLEEP AND RECOVERY Chapter 7: The Sci...,Chapter 8 discuses impruving sleep qualitee by...,single_hop_specifc_query_synthesizer
4,What practical strategies does Chapter 15 reco...,[PART 5: BUILDING HEALTHY HABITS Chapter 13: T...,Chapter 15 suggests several practical strategi...,single_hop_specifc_query_synthesizer
5,What practical strategies for wellness are rec...,[PART 5: BUILDING HEALTHY HABITS Chapter 13: T...,Chapter 14 suggests that starting your morning...,single_hop_specifc_query_synthesizer
6,How can strategies from Chapter 9 for managing...,[<1-hop>\n\nPART 3: SLEEP AND RECOVERY Chapter...,Chapter 9 explains that managing insomnia invo...,multi_hop_specific_query_synthesizer
7,According to the wellness strategies outlined ...,[<1-hop>\n\nPART 3: SLEEP AND RECOVERY Chapter...,"Improving sleep quality, as described in Chapt...",multi_hop_specific_query_synthesizer
8,How do the recommendations from Chapter 7 on s...,[<1-hop>\n\nPART 3: SLEEP AND RECOVERY Chapter...,Chapter 7 emphasizes the importance of sleep f...,multi_hop_specific_query_synthesizer
9,How can the sleep hygiene practices outlined i...,[<1-hop>\n\nPART 3: SLEEP AND RECOVERY Chapter...,The sleep hygiene practices described in Chapt...,multi_hop_specific_query_synthesizer


## Task 4: Construct our RAG application

Now we'll construct our LangChain RAG, which we will be evaluating using the above created test data!

### R - Retrieval

Let's start with building our retrieval pipeline, which will involve loading the same data we used to create our synthetic test set above.

> NOTE: We need to use the same data - as our test set is specifically designed for this data.

In [6]:
loader = TextLoader("data/HealthWellnessGuide.txt")
docs = loader.load()

Now that we have our data loaded, let's split it into chunks!

In [7]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=50, chunk_overlap=0)
split_documents = text_splitter.split_documents(docs)
len(split_documents)

447

### ❓ Question #1:

What is the purpose of the `chunk_overlap` parameter in the `RecursiveCharacterTextSplitter`?

##### Answer:

Chunk overlap is the number of characters shared between consecutive chunks. The overlap helps maintain context by ensuring information is not lost when text is being chunked.

Next up, we'll need to provide an embedding model that we can use to construct our vector store.

In [8]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

Now we can build our in memory QDrant vector store.

In [9]:
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams

client = QdrantClient(":memory:")

client.create_collection(
    collection_name="use_case_data",
    vectors_config=VectorParams(size=1536, distance=Distance.COSINE),
)

vector_store = QdrantVectorStore(
    client=client,
    collection_name="use_case_data",
    embedding=embeddings,
)

We can now add our documents to our vector store.

In [10]:
_ = vector_store.add_documents(documents=split_documents)

Let's define our retriever.

In [11]:
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

Now we can produce a node for retrieval!

In [12]:
def retrieve(state):
  retrieved_docs = retriever.invoke(state["question"])
  return {"context" : retrieved_docs}

### A - Augmented

Let's create a simple RAG prompt!

In [13]:
from langchain.prompts import ChatPromptTemplate

RAG_PROMPT = """\
You are a helpful assistant who answers questions based on provided context. You must only use the provided context, and cannot use your own knowledge.

### Question
{question}

### Context
{context}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_PROMPT)

### G - Generation

We'll also need an LLM to generate responses - we'll use `gpt-4o-nano` to avoid using the same model as our judge model.

In [14]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4.1-nano")

Then we can create a `generate` node!

In [15]:
def generate(state):
  docs_content = "\n\n".join(doc.page_content for doc in state["context"])
  messages = rag_prompt.format_messages(question=state["question"], context=docs_content)
  response = llm.invoke(messages)
  return {"response" : response.content}

### Building RAG Graph with LangGraph

Let's create some state for our LangGraph RAG graph!

In [16]:
from langgraph.graph import START, StateGraph
from typing_extensions import List, TypedDict
from langchain_core.documents import Document

class State(TypedDict):
  question: str
  context: List[Document]
  response: str

Now we can build our simple graph!

> NOTE: We're using `add_sequence` since we will always move from retrieval to generation. This is essentially building a chain in LangGraph.

In [17]:
graph_builder = StateGraph(State).add_sequence([retrieve, generate])
graph_builder.add_edge(START, "retrieve")
graph = graph_builder.compile()

Let's do a test to make sure it's doing what we'd expect.

In [18]:
response = graph.invoke({"question" : "What exercises help with lower back pain?"})

In [19]:
response["response"]

'The provided context does not specify specific exercises that help with lower back pain.'

## Task 5: Evaluating our Application with Ragas

Now we can finally do our evaluation!

We'll start by running the queries we generated usign SDG above through our application to get context and responses.

In [20]:
for test_row in dataset:
  response = graph.invoke({"question" : test_row.eval_sample.user_input})
  test_row.eval_sample.response = response["response"]
  test_row.eval_sample.retrieved_contexts = [context.page_content for context in response["context"]]

In [21]:
dataset.samples[0].eval_sample.response

'The Bird Dog exercise can be incorporated into a wellness routine by starting from a hands and knees position and then extending the opposite arm and leg. This movement helps strengthen the lower back muscles and improve stability, which can assist in relieving lower back pain. Incorporating this exercise regularly into your routine can support lower back health and reduce discomfort.'

Then we can convert that table into a `EvaluationDataset` which will make the process of evaluation smoother.

In [22]:
from ragas import EvaluationDataset

evaluation_dataset = EvaluationDataset.from_pandas(dataset.to_pandas())

We'll need to select a judge model - in this case we're using the same model that was used to generate our Synthetic Data.

In [23]:
from ragas import evaluate
from ragas.llms import LangchainLLMWrapper

evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini"))

Next up - we simply evaluate on our desired metrics!

In [24]:
from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness, ResponseRelevancy, ContextEntityRecall, NoiseSensitivity
from ragas import evaluate, RunConfig

custom_run_config = RunConfig(timeout=360)

baseline_result = evaluate(
    dataset=evaluation_dataset,
    metrics=[LLMContextRecall(), Faithfulness(), FactualCorrectness(), ResponseRelevancy(), ContextEntityRecall(), NoiseSensitivity()],
    llm=evaluator_llm,
    run_config=custom_run_config
)
baseline_result

Evaluating:   0%|          | 0/66 [00:00<?, ?it/s]

{'context_recall': 0.3030, 'faithfulness': 0.6856, 'factual_correctness': 0.2791, 'answer_relevancy': 0.3499, 'context_entity_recall': 0.3076, 'noise_sensitivity_relevant': 0.0646}

## Task 6: Making Adjustments and Re-Evaluating

Now that we've got our baseline - let's make a change and see how the model improves or doesn't improve!




We'll first set our retriever to return more documents, which will allow us to take advantage of the reranking.

In [25]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=30)
split_documents = text_splitter.split_documents(docs)
len(split_documents)

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

client = QdrantClient(":memory:")

client.create_collection(
    collection_name="use_case_data_new_chunks",
    vectors_config=VectorParams(size=1536, distance=Distance.COSINE),
)

vector_store = QdrantVectorStore(
    client=client,
    collection_name="use_case_data_new_chunks",
    embedding=embeddings,
)

_ = vector_store.add_documents(documents=split_documents)

adjusted_example_retriever = vector_store.as_retriever(search_kwargs={"k": 20})

Reranking, or contextual compression, is a technique that uses a reranker to compress the retrieved documents into a smaller set of documents.

This is essentially a slower, more accurate form of semantic similarity that we use on a smaller subset of our documents.

In [26]:
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_cohere import CohereRerank

def retrieve_adjusted(state):
  compressor = CohereRerank(model="rerank-v3.5")
  compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=adjusted_example_retriever, search_kwargs={"k": 5}
  )
  retrieved_docs = compression_retriever.invoke(state["question"])
  return {"context" : retrieved_docs}

We can simply rebuild our graph with the new retriever!

In [32]:
class AdjustedState(TypedDict):
  question: str
  context: List[Document]
  response: str

adjusted_graph_builder = StateGraph(AdjustedState).add_sequence([retrieve_adjusted, generate])
adjusted_graph_builder.add_edge(START, "retrieve_adjusted")
adjusted_graph = adjusted_graph_builder.compile()

In [33]:
response = adjusted_graph.invoke({"question" : "How can I improve my sleep quality?"})
response["response"]

ConnectError: [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: self-signed certificate in certificate chain (_ssl.c:1032)

In [ ]:
import time
import copy

rerank_dataset = copy.deepcopy(dataset)

for test_row in rerank_dataset:
  response = adjusted_graph.invoke({"question" : test_row.eval_sample.user_input})
  test_row.eval_sample.response = response["response"]
  test_row.eval_sample.retrieved_contexts = [context.page_content for context in response["context"]]
  time.sleep(5) # To try to avoid rate limiting.

In [ ]:
rerank_dataset.samples[0].eval_sample.response

'To help with lower back pain, gentle stretching and strengthening exercises are recommended. You can try the Cat-Cow Stretch, which involves starting on your hands and knees and alternately arching your back up (cat) and letting it sag down (cow), doing 10-15 repetitions. Another good exercise is the Bird Dog, where from hands and knees, you extend opposite arm and leg, hold for 5 seconds, and switch sides, completing 10 repetitions per side. Additionally, Pelvic Tilts can help; lie on your back with knees bent, tighten your abs and tilt your pelvis up slightly to flatten your back against the floor, hold for 10 seconds, and repeat 8-12 times. These exercises can help relieve lower back pain and prevent future issues.'

In [ ]:
rerank_evaluation_dataset = EvaluationDataset.from_pandas(rerank_dataset.to_pandas())

In [ ]:
rerank_result = evaluate(
    dataset=rerank_evaluation_dataset,
    metrics=[LLMContextRecall(), Faithfulness(), FactualCorrectness(), ResponseRelevancy(), ContextEntityRecall(), NoiseSensitivity()],
    llm=evaluator_llm,
    run_config=custom_run_config
)
rerank_result

Evaluating:   0%|          | 0/66 [00:00<?, ?it/s]

{'context_recall': 0.6667, 'faithfulness': 0.8240, 'factual_correctness': 0.7273, 'answer_relevancy': 0.7805, 'context_entity_recall': 0.3247, 'noise_sensitivity_relevant': 0.1393}

### ❓ Question #2:

Which system performed better, on what metrics, and why?

##### Answer:

The reranked version performed better on all of metrics because it reads the query and the document together and decides if the document is relevant to the specific query which ensures better semantic interactions. On the other hand, embedding retrieval retrieves by approximate similarity (cosine similarity between embedded vector and document) which is fast but less reliable. 

E.g. query: "Apple revenue last year" -> Embedding might retrieve text about agriculture which is totaly irelevant.

### ❓ Question #3:

What are the benefits and limitations of using synthetic data generation for RAG evaluation? Consider both the practical advantages and potential pitfalls.

##### Answer:

Benefits are scalabiliy and speed, potential coverage of edge cases. Also it enables calculation of metrics and helps in domains where it is hard or impossible to get labeled data. 
Limitations include unrealistic distribution (often clean queries when in reallity they are not), model bias (using llm to create data and similar llm in our system) when avaluating. Furthermore, sintetic queries  often closely match style of text in docuemnt which may lead to retrieval appearing better than it actually is.   

### ❓ Question #4:

If you were building a production wellness assistant, which Ragas metrics would be most important to optimize for and why? Consider the healthcare/wellness domain specifically.

##### Answer:

For a production wellness assistant, the most important Ragas metrics to optimize would be those that protect safety, factual accuracy, and appropriate use of retrieved evidence, since healthcare and wellness advice can directly impact user well-being. The top priority would be faithfulness. In healthcare, hallucinated or unsupported claims can cause harm. The assistant must base its recommendations strictly on retrieved evidence or verified knowledge. Optimizing for faithfulness ensures the model does not invent studies, exaggerate findings, or provide unsafe medical advice that is not grounded in sources. Another crucial one i would target is answer relevancy. Wellness users often ask targeted, personal questions (e.g., “Is this safe with my medication?”). The system must directly address the user’s concern rather than provide generic health information. High relevancy reduces the risk of missing critical context.

## Activity #1: Implement a Different Reranking Strategy

In this activity, you'll experiment with different reranking parameters or strategies to see how they affect the evaluation metrics.

**Requirements:**
1. Modify the `retrieve_adjusted` function to use different parameters (e.g., change `k` values, try different top_n for reranking)
2. Or implement a different retrieval enhancement strategy (e.g., hybrid search, query expansion)
3. Run the evaluation and compare results with the baseline and reranking results above
4. Document your findings in the markdown cell below

In [ ]:
### YOUR CODE HERE ###

# Implement your custom retrieval strategy here
# Example: modify retrieve_adjusted with different parameters

query_expansion_prompt = ChatPromptTemplate.from_template("""\
You are a helpful assistant who expands a user's question to be more detailed and specific, in order to improve retrieval results. 
Given the user's original question, provide an expanded version of the question that includes more context and details. 
The expanded question should be designed to retrieve more relevant documents from a knowledge base.

Original query: {query}

Expanded query:"""
)

def expand_query(state):
    messages = query_expansion_prompt.format_messages(query=state["question"])
    response = llm.invoke(messages)
    return {"question" : response.content}

expanded_graph_builder = StateGraph(AdjustedState).add_sequence([expand_query, retrieve_adjusted, generate])
expanded_graph_builder.add_edge(START, "expand_query")
expanded_graph = expanded_graph_builder.compile()
response = expanded_graph.invoke({"question" : "What are some effective strategies for managing stress?"})
response["response"]

'Effective management of stress in daily life can be achieved through a combination of evidence-based strategies and practical techniques. For immediate relief, practices such as deep breathing (inhale for 4 counts, hold for 4, exhale for 4), progressive muscle relaxation, grounding techniques (naming things you see, hear, feel, smell, taste), taking short walks in nature, and listening to calming music can help reduce acute stress. \n\nFor long-term stress reduction, incorporating regular exercise and ensuring adequate sleep are highly beneficial. Engaging with social connections and support systems provides emotional stability. Effective time management and prioritization help prevent feeling overwhelmed and promote a sense of control. Setting healthy boundaries in both work and personal life is essential to manage stressors effectively. Leisure activities and hobbies offer relaxation and joy, counteracting stress.\n\nMindfulness and meditation are also valuable tools; practicing min

In [ ]:
# evaluate the new retrieval strategy on the dataset
expanded_dataset = copy.deepcopy(dataset)

for test_row in expanded_dataset:
  response = expanded_graph.invoke({"question" : test_row.eval_sample.user_input})
  test_row.eval_sample.response = response["response"]
  test_row.eval_sample.retrieved_contexts = [context.page_content for context in response["context"]]
  time.sleep(5) # To try to avoid rate limiting.

In [ ]:
expanded_evaluation_dataset = EvaluationDataset.from_pandas(expanded_dataset.to_pandas())
expanded_result = evaluate(
    dataset=expanded_evaluation_dataset,
    metrics=[LLMContextRecall(), Faithfulness(), FactualCorrectness(), ResponseRelevancy(), ContextEntityRecall(), NoiseSensitivity()],
    llm=evaluator_llm,
    run_config=custom_run_config
)
expanded_result

Evaluating:   0%|          | 0/66 [00:00<?, ?it/s]

{'context_recall': 0.7121, 'faithfulness': 0.5923, 'factual_correctness': 0.6118, 'answer_relevancy': 0.6875, 'context_entity_recall': 0.3769, 'noise_sensitivity_relevant': 0.1317}

### Activity #1 Findings:

*Document your findings here: What strategy did you try? How did it compare to the baseline and reranking results?*